<a href="https://colab.research.google.com/github/JordanTerwilliger/Intro-to-Deep-Learning/blob/main/HW3/HW3_Seq2Seq_Q1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Jordan Terwilliger, 801343938, HW3

#Imports

In [35]:
import torch
import torch.nn as nn
import torch.functional as F

import matplotlib.pyplot as plt
import numpy as np

import requests

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

try:
  import torchinfo
except ImportError:
  !pip install torchinfo
  import torchinfo

from torchinfo import summary

from torch import optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [36]:
url = "https://raw.githubusercontent.com/JordanTerwilliger/Intro-to-Deep-Learning/refs/heads/main/HW3/vast_english_french.txt"
response = requests.get(url)
text = response.text  # This is the entire text data

#Preprocessing

In [37]:
def tokenize_and_pad(sentences, vocab):
    # Calculate the maximum sentence length for padding
    max_length = max(len(sentence.split(' ')) for sentence in sentences) + 2  # +2 for SOS and EOS tokens
    tokenized_sentences = []
    for sentence in sentences:
        # Convert each sentence to a list of indices, adding SOS and EOS tokens
        tokens = [vocab.word2index["<SOS>"]] + [vocab.word2index[word] for word in sentence.split(' ')] + [vocab.word2index["<EOS>"]]
        # Pad sentences to the maximum length
        padded_tokens = tokens + [vocab.word2index["<PAD>"]] * (max_length - len(tokens))
        tokenized_sentences.append(padded_tokens)
    return torch.tensor(tokenized_sentences, dtype=torch.long)

In [38]:
# Vocabulary class to handle mapping between words and numerical indices
class Vocabulary:
  def __init__(self):
    #Dictionaries for special tokens and reverse
    self.word2index = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2}
    self.index2word = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>"}
    self.word_count = {}  # Keep track of word frequencies
    self.n_words = 3  # Start counting from 3 to account for special tokens

  def add_sentence(self, sentence):
    # Add all words in a sentence to the vocabulary
    for word in sentence.split(' '):
      self.add_word(word)

  def add_word(self, word):
        # Add a word to the vocabulary
        if word not in self.word2index:
            # Assign a new index to the word and update mappings
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.word_count[word] = 1
            self.n_words += 1
        else:
            # Increment word count if the word already exists in the vocabulary
            self.word_count[word] += 1

In [39]:
class EngFrDataset(Dataset):
    def __init__(self, pairs):
        self.eng_vocab = Vocabulary()
        self.fr_vocab = Vocabulary()
        self.pairs = []

        # Process each English-French pair
        for eng, fr in pairs:
            self.eng_vocab.add_sentence(eng)
            self.fr_vocab.add_sentence(fr)
            self.pairs.append((eng, fr))

        # Separate English and French sentences
        self.eng_sentences = [pair[0] for pair in self.pairs]
        self.fr_sentences = [pair[1] for pair in self.pairs]

        # Tokenize and pad sentences
        self.eng_tokens = tokenize_and_pad(self.eng_sentences, self.eng_vocab)
        self.fr_tokens = tokenize_and_pad(self.fr_sentences, self.fr_vocab)

    def __len__(self):
        # Return the number of sentence pairs
        return len(self.pairs)

    def __getitem__(self, idx):
        # Get the tokenized and padded sentences by index
        eng_tokens = self.eng_tokens[idx]
        fr_tokens = self.fr_tokens[idx]
        return eng_tokens, fr_tokens

In [40]:
pairs = [
    tuple(part.strip() for part in line.split('\t')) for line in text.splitlines() if '\t' in line
]

dataset = EngFrDataset(pairs)

train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

idx2w_e = dataset.eng_vocab.index2word
idx2w_f = dataset.fr_vocab.index2word
batch_size = 1
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)


#Model Creation

In [41]:
class Encoder(nn.Module):
  def __init__(self, input_size, hidden_size):
    super(Encoder,self).__init__()
    self.hidden_size = hidden_size
    self.embedding = nn.Embedding(input_size, hidden_size) #num_embeddings, embedding_dim
    self.dropout = nn.Dropout(p = 0.2)
    self.rnn = nn.GRU(hidden_size, hidden_size)

  def initHidden(self):
      # Shape: (num_layers, batch_size=1, hidden_size)
      return torch.zeros(1, 1, self.hidden_size, device=device)

  def forward(self, input_tensor, hidden_state):
    embedded = self.embedding(input_tensor).view(1, 1, -1)
    output = self.dropout(embedded)
    output, hidden = self.rnn(output, hidden_state)
    return output, hidden #Hidden is the input to the decoder, Input -> Encoder -> Encoder Output + Hidden State -> Decoder -> Output

class Decoder(nn.Module):
  def __init__(self, hidden_size, output_size):
    super(Decoder, self).__init__()
    self.hidden_size = hidden_size
    self.embedding = nn.Embedding(output_size, hidden_size) #num_embeddings, embedding_dim
    self.dropout = nn.Dropout(p = 0.2)
    self.rnn = nn.GRU(hidden_size, hidden_size)
    self.out = nn.Linear(hidden_size, output_size)
    self.softmax = nn.LogSoftmax(dim=1)

  def forward(self, input_tensor, hidden_state):
    embedded = self.embedding(input_tensor).view(1,1,-1)
    output = self.dropout(embedded)
    output, hidden_state = self.rnn(output, hidden_state)
    output = self.softmax(self.out(output.squeeze(0)))
    return output, hidden_state

In [42]:
input_size = len(dataset.eng_vocab.index2word)
hidden_size = 256
output_size = len(dataset.fr_vocab.index2word)

encoder = Encoder(input_size=input_size, hidden_size=hidden_size).to(device)
decoder = Decoder(hidden_size=hidden_size, output_size=output_size).to(device)

In [43]:
learning_rate = 0.01
encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

#Training Loop


In [44]:
PAD = 0
SOS = 1
EOS = 2

def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion):
  encoder.train()
  decoder.train()
  encoder_hidden = encoder.initHidden()
  encoder_optimizer.zero_grad()
  decoder_optimizer.zero_grad()

  input_length = input_tensor.size(0)
  target_length = target_tensor.size(0)
  loss = 0

  for ei in range(input_length):
    encoder_output, encoder_hidden = encoder(input_tensor[ei].unsqueeze(0), encoder_hidden) #Encoder gets input and hidden state

  decoder_input = torch.tensor([[SOS]], device = device)
  decoder_hidden = encoder_hidden #Output of encoder goes to input of decoder


  for di in range(target_length):
    decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
    _, topi = decoder_output.topk(1)
    decoder_input = topi.squeeze().detach()

    loss += criterion(decoder_output, target_tensor[di].unsqueeze(0))
    if decoder_input.item() == EOS:
      break

  loss.backward()
  encoder_optimizer.step()
  decoder_optimizer.step()

  return loss.item() / target_length

In [45]:
def test(input_tensor, target_tensor, encoder, decoder, criterion):
  encoder_hidden = encoder.initHidden()
  encoder.eval()
  decoder.eval()
  input_length = input_tensor.size(0)
  target_length = target_tensor.size(0)
  val_loss = 0

  correct = 0

  with torch.no_grad():
      for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(input_tensor[ei].unsqueeze(0), encoder_hidden)
      decoder_input = torch.tensor([[SOS]], device = device)
      decoder_hidden = encoder_hidden

      for di in range(target_length):
        decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
        _, topi = decoder_output.topk(1)
        decoder_input = topi.squeeze().detach()

        val_loss += criterion(decoder_output, target_tensor[di].unsqueeze(0))
        if(decoder_input.item() == target_tensor[di].item()):
          correct += 1
        if decoder_input.item() == EOS:
          break



      return (val_loss.item() / target_length), (correct / target_length)

In [46]:
epochs = 30

print("Starting Training...")
for epoch in range(epochs):
  total_loss = 0
  total_val_loss = 0
  total_val_accuracy = 0

  for input_tensor, target_tensor in train_loader:
    input_tensor = input_tensor[0].to(device)
    target_tensor = target_tensor[0].to(device)
    loss = train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
    total_loss += loss

  for input_tensor, target_tensor in test_loader:
    input_tensor = input_tensor[0].to(device)
    target_tensor = target_tensor[0].to(device)
    val_loss, val_accuracy = test(input_tensor, target_tensor, encoder, decoder, criterion)
    total_val_loss += val_loss
    total_val_accuracy += val_accuracy

  print(f"Epoch {epoch}, Train Loss: {(total_loss / len(train_loader)):.4f}, Val Loss: {(total_val_loss / len(test_loader)):.4f}, Val Accuracy: {(total_val_accuracy / len(test_loader)):.4f}")

Starting Training...
Epoch 0, Train Loss: 1.8195, Val Loss: 1.3345, Val Accuracy: 0.0822
Epoch 1, Train Loss: 1.6955, Val Loss: 1.3354, Val Accuracy: 0.0777
Epoch 2, Train Loss: 1.6526, Val Loss: 1.7080, Val Accuracy: 0.0811
Epoch 3, Train Loss: 1.6537, Val Loss: 1.6617, Val Accuracy: 0.0816
Epoch 4, Train Loss: 1.7719, Val Loss: 1.8528, Val Accuracy: 0.0884
Epoch 5, Train Loss: 1.8978, Val Loss: 1.8056, Val Accuracy: 0.0907
Epoch 6, Train Loss: 1.8353, Val Loss: 1.8450, Val Accuracy: 0.0811
Epoch 7, Train Loss: 1.8368, Val Loss: 2.1380, Val Accuracy: 0.0957
Epoch 8, Train Loss: 1.8268, Val Loss: 2.0907, Val Accuracy: 0.1171
Epoch 9, Train Loss: 1.7865, Val Loss: 1.9994, Val Accuracy: 0.1244
Epoch 10, Train Loss: 1.7173, Val Loss: 1.9339, Val Accuracy: 0.1278
Epoch 11, Train Loss: 1.6358, Val Loss: 1.9567, Val Accuracy: 0.1329
Epoch 12, Train Loss: 1.6206, Val Loss: 1.9224, Val Accuracy: 0.1295
Epoch 13, Train Loss: 1.5376, Val Loss: 1.9271, Val Accuracy: 0.1374
Epoch 14, Train Loss: 1

In [49]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def BLEU(encoder, decoder, dataloader, n_examples = 5):
  total_bleu_score = 0.0
  chencherry = SmoothingFunction()  # Smooths BLEU variations for very short sequences


  with torch.no_grad():
    for i, (input_tensor, target_tensor) in enumerate(dataloader):
      input_tensor = input_tensor[0].to(device)
      target_tensor = target_tensor[0].to(device)

      encoder_hidden = encoder.initHidden()
      input_length = input_tensor.size(0)

       # Pass through Encoder
      for ei in range(input_length):
          encoder_output, encoder_hidden = encoder(input_tensor[ei].unsqueeze(0), encoder_hidden)

      # Setup Decoder
      decoder_input = torch.tensor([[SOS]], device=device)
      decoder_hidden = encoder_hidden
      predicted_indices = []

      # Generate step
      for di in range(20):  # Hard cap sequence generation length
          decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
          _, topi = decoder_output.topk(1)
          idx_pred = topi.item()
          predicted_indices.append(idx_pred)
          decoder_input = topi.squeeze().detach()

          if idx_pred == EOS:
              break

      # Convert Index arrays back into readable raw character strings
      input_string = ' '.join([idx2w_e[idx.item()] for idx in input_tensor if idx.item() not in (SOS, EOS)])
      target_string = ' '.join([idx2w_f[idx.item()] for idx in target_tensor if idx.item() not in (SOS, EOS)])
      predicted_string = ' '.join([idx2w_f[idx] for idx in predicted_indices if idx not in (SOS, EOS)])

      # --- EVALUATION 2: BLEU Score (Generative Overlap Metric) ---
      reference_tokens = [target_string.split()]  # References must be a list of token lists
      candidate_tokens = predicted_string.split()  # Candidate is a single token list

      bleu = sentence_bleu(reference_tokens, candidate_tokens, smoothing_function=chencherry.method1)
      total_bleu_score += bleu


      if i < n_examples:
        match_status = "PASS" if predicted_string == target_string else "FAIL"
        print(f'Input: {input_string:<12} | Target: {target_string:<12} | Predicted: {predicted_string:<12} | Match: {match_status:<4} | BLEU: {bleu:.4f}')


    average_bleu = total_bleu_score / len(dataloader)

    print(f' -> Average Validation BLEU-4 Score : {average_bleu:.4f}')

In [48]:
BLEU(encoder, decoder, test_loader)

Input: She bought a beautiful painting <PAD> <PAD> <PAD> <PAD> <PAD> | Target: Elle a acheté un beau tableau <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> | Predicted: Elle a gagné un tournoi | Match: FAIL | BLEU: 0.0208
Input: She wears a beautiful gold ring <PAD> <PAD> <PAD> <PAD> | Target: Elle porte une belle bague en or <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> | Predicted: Elle a acheté un sac en cuir | Match: FAIL | BLEU: 0.0145
Input: They live in a beautiful apartment facing the river <PAD> | Target: Ils vivent dans un bel appartement face à la rivière <PAD> <PAD> <PAD> <PAD> | Predicted: Ils ont construit une grand mur confortable à la à | Match: FAIL | BLEU: 0.0331
Input: They share their secrets <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> | Target: Ils partagent leurs secrets <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> | Predicted: Ils parlent espagnol | Match: FAIL | BLEU: 0.0029
Input: The dog barks loudly <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> | Target: Le chien a